In [ ]:
# --- Cell 1: environment + scratch space discovery -------------------------
import os, sys, platform, subprocess

print("project:", "Edge0-120B")
print("python:", sys.version.split()[0])
print("platform:", platform.platform())

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for p in ("/kaggle/temp", "/kaggle/working"):
    os.makedirs(p, exist_ok=True)
    st = os.statvfs(p)
    free = st.f_bavail * st.f_frsize / 1e9
    print(f"{p}: free={free:.1f} GB writable={os.access(p, os.W_OK)}")

print("--- GPU ---")
print(subprocess.run(["nvidia-smi",
                      "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "no nvidia-smi")

print("--- internet ---")
for u in ("https://huggingface.co", "https://github.com", "https://gh-proxy.com"):
    r = subprocess.run(["curl", "-4", "-s", "-o", "/dev/null", "-w", "%{http_code}",
                        "--connect-timeout", "5", "--max-time", "10", u], capture_output=True, text=True)
    print(u, "->", r.stdout)


In [ ]:
# --- Cell 2: Edge0 preflight (no llama.cpp here) ---------------------------
import os, shutil, subprocess

print("model:", "Edge0-120B")
print("pack :", "Edge0/Edge0-120B-q3k")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
free = shutil.disk_usage("/kaggle/temp").free / 1e9
nvcc = shutil.which("nvcc") or "/usr/local/cuda/bin/nvcc"
print(gpu or "no gpu")
print(f"RAM {ram:.0f} GB | /kaggle/temp free {free:.0f} GB | nvcc {nvcc}")

print("""
Edge0 needs:
  * the Strata engine built from github.com/Edge0-AI/Edge0 (CUDA 13 recipe;
    sm_75 is supported, but the build takes ~20-40 min here)
  * a 92-105 GB pack (use the -q3k repo, not -NVFP4: T4 has no FP4 hardware)
  * >= 16 GiB VRAM for one GPU + ~20 GiB RAM for the expert cache
Expected on 2x T4 (estimate): 5-15 tok/s decode, 80-250 tok/s prefill.
This kernel only reports the verdict; see edge0-*/README.md for the manual
recipe and expected numbers.
""", flush=True)
print("PREFLIGHT_DONE", flush=True)
